In [23]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath('..'))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import pandas as pd
from relbench.datasets.f1 import F1Dataset
from showcase_helper import ConverterShowcaseHelper

#### TEMPORAL


In [25]:
f1_dataset = F1Dataset()
timestamps = pd.date_range(start="2010-01-01", end="2010-05-05", freq="D")

db = f1_dataset.get_db(upto_test_timestamp=False)
temporal_helper = ConverterShowcaseHelper(db, timestamps)

Making Database object from scratch...
(You can also use `get_dataset(..., download=True)` for datasets prepared by the RelBench team.)
Done in 0.31 seconds.


In [28]:
temporal_helper.convert_query("""
                PREDICT AVG(reuls.position, 0, 10, days) CONTAINS "L"
                FOR EACH drivers.driverId 
                ASSUMING SUM(results.position, 0, 10, days) > 100
                              WHERE AVG(results.position, 
                -365, 0, das) < 30;
              """)


                PREDICT AVG(reuls.position, 0, 10, days) CONTAINS "L"
                FOR EACH drivers.driverId 
                ASSUMING SUM(results.position, 0, 10, days) > 100
                              WHERE AVG(results.position, 
                -365, 0, das) < 30;
              


1. ERROR at line 6:25 - mismatched input 'das' expecting TIME_MEASURE_UNIT
2. ERROR at line 2:28 - Table 'reuls' in temporal aggregation does not exist in database
3. ERROR at line 2:28 - Table 'reuls' in temporal aggregation is not connected to main table 'drivers'
4. ERROR at line 2:28 - Table 'reuls' in temporal aggregation does not have a time column
5. ERROR at line 2:34 - Column 'position' in temporal aggregation does not exist in table 'reuls'
6. ERROR at line 2:24 - Aggregation type 'AVG' cannot be used in string condition
7. ERROR at line 4:47 - Start and end time in temporal aggregation must be non-positive in ASSUMING clause, found start=0.0, end=10.0
8. ERROR at line 6:16 - Start and end time in temporal aggregation must be non-negative in PREDICT and WHERE clauses, found start=-365.0, end=0.0


SystemExit: 1

In [ ]:
temporal_helper.convert_query("""
    PREDICT AVG(results.positionOrder WHERE AVG(results.positionOrder) == 20, 0, 60, DAYS)
    FOR EACH drivers.driverId;
""")


    PREDICT AVG(results.positionOrder WHERE AVG(results.positionOrder) == 20, 0, 60, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 0, 60, DAYS) IS NOT NULL;

SELECT
    *
FROM
  (------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        main.comp_col
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
    

In [23]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 100, days) > 10
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(results.position, 0, 100, days) > 10
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01  False
1         1 2010-01-01  False
2         2 2010-01-01  False
3         3 2010-01-01  False
4         4 2010-01-01   True
...     ...        ...    ...
107120  852 2010-05-05  False
107121  853 2010-05-05  False
107122  854 2010-05-05  False
107123  855 2010-05-05  False
107124  856 2010-05-05  False

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [24]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              


------------------ Table ------------------
DataFrame:
         fk  timestamp      label
0         0 2010-01-01   5.000000
1         1 2010-01-01        NaN
2         2 2010-01-01   4.333333
3         3 2010-01-01   6.000000
4         4 2010-01-01  14.000000
...     ...        ...        ...
107120  852 2010-05-05        NaN
107121  853 2010-05-05        NaN
107122  854 2010-05-05        NaN
107123  855 2010-05-05        NaN
107124  856 2010-05-05        NaN

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [25]:
temporal_helper.convert_query("""
                PREDICT NOT AVG(results.position, 0, 100, days) > 10
                FOR EACH drivers.driverId;
              """)


                PREDICT NOT AVG(results.position, 0, 100, days) > 10
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01   True
1         1 2010-01-01  False
2         2 2010-01-01   True
3         3 2010-01-01   True
4         4 2010-01-01  False
...     ...        ...    ...
107120  852 2010-05-05  False
107121  853 2010-05-05  False
107122  854 2010-05-05  False
107123  855 2010-05-05  False
107124  856 2010-05-05  False

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [26]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 700, days) > 10
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(results.position, 0, 700, days) > 10
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01  False
1         1 2010-01-01  False
2         2 2010-01-01  False
3         3 2010-01-01  False
4         4 2010-01-01   True
...     ...        ...    ...
107120  852 2010-05-05  False
107121  853 2010-05-05  False
107122  854 2010-05-05  False
107123  855 2010-05-05  False
107124  856 2010-05-05  False

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [27]:
temporal_helper.convert_query("""
                PREDICT NOT AVG(results.position, 0, 700, days) > 10
                FOR EACH drivers.driverId;
              """)


                PREDICT NOT AVG(results.position, 0, 700, days) > 10
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01   True
1         1 2010-01-01   True
2         2 2010-01-01   True
3         3 2010-01-01   True
4         4 2010-01-01  False
...     ...        ...    ...
107120  852 2010-05-05  False
107121  853 2010-05-05  False
107122  854 2010-05-05  False
107123  855 2010-05-05  False
107124  856 2010-05-05  False

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [ ]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 10, days) > 10
                FOR EACH drivers.driverId 
                WHERE AVG(results.position, -365, 0, days) < 30 AND AVG(results.position, -365, 0, days) >= 0;
              """)


                PREDICT AVG(results.position, 0, 10, days) > 10
                FOR EACH drivers.driverId WHERE AVG(results.position, -365, 0, days) < 30 AND AVG(results.position, -365, 0, days) >= 0;
              


1. ERROR at line 3:70 - Start and end time in temporal aggregation must be non-negative in PREDICT and WHERE clauses, found start=-365, end=0
2. ERROR at line 3:116 - Start and end time in temporal aggregation must be non-negative in PREDICT and WHERE clauses, found start=-365, end=0


SystemExit: 1

/home/kolesiko/CTU/BT/RTGL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [30]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 100, days) < 100
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(results.position, 0, 100, days) < 100
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01   True
1         1 2010-01-01  False
2         2 2010-01-01   True
3         3 2010-01-01   True
4         4 2010-01-01   True
...     ...        ...    ...
107120  852 2010-05-05  False
107121  853 2010-05-05  False
107122  854 2010-05-05  False
107123  855 2010-05-05  False
107124  856 2010-05-05  False

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [31]:
temporal_helper.convert_query("""
                PREDICT COUNT(results.*, 0, 100, days) < 100
                FOR EACH drivers.driverId;
              """)


                PREDICT COUNT(results.*, 0, 100, days) < 100
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01   True
1         1 2010-01-01   True
2         2 2010-01-01   True
3         3 2010-01-01   True
4         4 2010-01-01   True
...     ...        ...    ...
107120  852 2010-05-05   True
107121  853 2010-05-05   True
107122  854 2010-05-05   True
107123  855 2010-05-05   True
107124  856 2010-05-05   True

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [32]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp      label
0         0 2010-01-01   5.000000
1         1 2010-01-01        NaN
2         2 2010-01-01   4.333333
3         3 2010-01-01   6.000000
4         4 2010-01-01  14.000000
...     ...        ...        ...
107120  852 2010-05-05        NaN
107121  853 2010-05-05        NaN
107122  854 2010-05-05        NaN
107123  855 2010-05-05        NaN
107124  856 2010-05-05        NaN

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [33]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position WHERE drivers.driverId < 5, 0, 100, days)
                FOR EACH drivers.driverId WHERE COUNT(results.*, 0, 100, days) > 5
                WHERE AVG(results.position, -365, 0, days) < 30;
              """)


                PREDICT AVG(results.position WHERE drivers.driverId < 5, 0, 100, days)
                FOR EACH drivers.driverId WHERE COUNT(results.*, 0, 100, days) > 5
                WHERE AVG(results.position, -365, 0, days) < 30;
              


1. ERROR at line 4:16 - mismatched input 'WHERE' expecting ';'
2. ERROR at line 2:51 - Table 'drivers' in condition is not connected to main table 'results'


SystemExit: 1

/home/kolesiko/CTU/BT/RTGL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [34]:
temporal_helper.convert_query("""
                PREDICT LIST_DISTINCT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT LIST_DISTINCT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp             label
0         0 2010-01-01        [6.0, 3.0]
1         1 2010-01-01              <NA>
2         2 2010-01-01        [5.0, 3.0]
3         3 2010-01-01  [4.0, 1.0, 13.0]
4         4 2010-01-01  [15.0, --, 13.0]
...     ...        ...               ...
107120  852 2010-05-05              <NA>
107121  853 2010-05-05              <NA>
107122  854 2010-05-05              <NA>
107123  855 2010-05-05              <NA>
107124  856 2010-05-05              <NA>

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [37]:
temporal_helper.convert_query("""
                PREDICT first(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT first(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01    3.0
1         1 2010-01-01    NaN
2         2 2010-01-01    5.0
3         3 2010-01-01    1.0
4         4 2010-01-01   15.0
...     ...        ...    ...
107120  852 2010-05-05    NaN
107121  853 2010-05-05    NaN
107122  854 2010-05-05    NaN
107123  855 2010-05-05    NaN
107124  856 2010-05-05    NaN

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [38]:
temporal_helper.convert_query("""
                PREDICT LIST_DISTINCT(results.position, 0, 100, days) CLASSIFY
                FOR EACH drivers.driverId;
              """)


                PREDICT LIST_DISTINCT(results.position, 0, 100, days) CLASSIFY
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp             label
0         0 2010-01-01        [6.0, 3.0]
1         1 2010-01-01              <NA>
2         2 2010-01-01        [5.0, 3.0]
3         3 2010-01-01  [1.0, 13.0, 4.0]
4         4 2010-01-01  [15.0, 13.0, --]
...     ...        ...               ...
107120  852 2010-05-05              <NA>
107121  853 2010-05-05              <NA>
107122  854 2010-05-05              <NA>
107123  855 2010-05-05              <NA>
107124  856 2010-05-05              <NA>

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [39]:
temporal_helper.convert_query("""
                PREDICT LIST_DISTINCT(results.position, 0, 100, days) RANK TOP 2
                FOR EACH drivers.driverId;
              """)


                PREDICT LIST_DISTINCT(results.position, 0, 100, days) RANK TOP 2
                FOR EACH drivers.driverId;
              


------------------ Table ------------------
DataFrame:
         fk  timestamp        label
0         0 2010-01-01   [6.0, 3.0]
1         1 2010-01-01         <NA>
2         2 2010-01-01   [5.0, 3.0]
3         3 2010-01-01  [13.0, 1.0]
4         4 2010-01-01   [15.0, --]
...     ...        ...          ...
107120  852 2010-05-05         <NA>
107121  853 2010-05-05         <NA>
107122  854 2010-05-05         <NA>
107123  855 2010-05-05         <NA>
107124  856 2010-05-05         <NA>

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [40]:
temporal_helper.convert_query("""
                PREDICT LIST_DISTINCT(results.position, 0, 100, days) RANK TOP 3
                FOR EACH drivers.driverId;
              """)


                PREDICT LIST_DISTINCT(results.position, 0, 100, days) RANK TOP 3
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp             label
0         0 2010-01-01        [6.0, 3.0]
1         1 2010-01-01              <NA>
2         2 2010-01-01        [5.0, 3.0]
3         3 2010-01-01  [4.0, 1.0, 13.0]
4         4 2010-01-01  [15.0, --, 13.0]
...     ...        ...               ...
107120  852 2010-05-05              <NA>
107121  853 2010-05-05              <NA>
107122  854 2010-05-05              <NA>
107123  855 2010-05-05              <NA>
107124  856 2010-05-05              <NA>

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [41]:
temporal_helper.convert_query("""
                PREDICT FIRST(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT FIRST(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01    3.0
1         1 2010-01-01    NaN
2         2 2010-01-01    5.0
3         3 2010-01-01    1.0
4         4 2010-01-01   15.0
...     ...        ...    ...
107120  852 2010-05-05    NaN
107121  853 2010-05-05    NaN
107122  854 2010-05-05    NaN
107123  855 2010-05-05    NaN
107124  856 2010-05-05    NaN

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [42]:
temporal_helper.convert_query("""
                PREDICT LIST_DISTINCT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT LIST_DISTINCT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp             label
0         0 2010-01-01        [6.0, 3.0]
1         1 2010-01-01              <NA>
2         2 2010-01-01        [5.0, 3.0]
3         3 2010-01-01  [4.0, 1.0, 13.0]
4         4 2010-01-01  [15.0, --, 13.0]
...     ...        ...               ...
107120  852 2010-05-05              <NA>
107121  853 2010-05-05              <NA>
107122  854 2010-05-05              <NA>
107123  855 2010-05-05              <NA>
107124  856 2010-05-05              <NA>

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [43]:
temporal_helper.convert_query("""
                PREDICT COUNT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              """)


                PREDICT COUNT(results.position, 0, 100, days)
                FOR EACH drivers.driverId;
              
------------------ Table ------------------
DataFrame:
         fk  timestamp  label
0         0 2010-01-01      3
1         1 2010-01-01      0
2         2 2010-01-01      3
3         3 2010-01-01      3
4         4 2010-01-01      2
...     ...        ...    ...
107120  852 2010-05-05      0
107121  853 2010-05-05      0
107122  854 2010-05-05      0
107123  855 2010-05-05      0
107124  856 2010-05-05      0

[107125 rows x 3 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'drivers'}
Primary Key Column: None
Time Column: timestamp
-------------------------------------------


In [15]:
temporal_helper.convert_query("""
                PREDICT AVG(results.position, 0, 100, days) CONTAINS "L"
                FOR EACH drivers.driverId WHERE drivers.driverId < 10;
              """)


                PREDICT AVG(results.position, 0, 100, days) CONTAINS "L"
                FOR EACH drivers.driverId WHERE drivers.driverId < 10;
              


1. ERROR at line 2:24 - Aggregation type 'AVG' cannot be used in string condition


SystemExit: 1

/home/kolesiko/CTU/BT/BT/RTGL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#### STATIC


In [29]:
f1_dataset = F1Dataset()
db = f1_dataset.get_db(upto_test_timestamp=False)
static_helper = ConverterShowcaseHelper(db, timestamps=None)

Making Database object from scratch...
(You can also use `get_dataset(..., download=True)` for datasets prepared by the RelBench team.)
Done in 0.34 seconds.


In [2]:
print(db.table_dict["drivers"])

NameError: name 'db' is not defined

In [34]:
static_helper.convert_query("""
                PREDICT AVG(drivers.surname) CONTAINS "L"
                FOR EACH drivr.driverId
                WHER drivers.driverId < 10;
              """)


                PREDICT AVG(drivers.surname) CONTAINS "L"
                FOR EACH drivr.driverId
                WHER drivers.driverId < 10;
              


1. ERROR at line 4:40 - mismatched input '10' expecting {NUM_COMP_OP, STR_COMP_OP, NULL_CHECK_OP}
2. ERROR at line 4:16 - mismatched input 'WHER' expecting {WHERE, ';'}
3. ERROR at line 3:25 - Table 'drivr' in FOR EACH clause does not exist in database
4. ERROR at line 3:31 - Column 'driverId' in FOR EACH clause does not exist in table 'drivr'
5. ERROR at line 3:31 - Column 'driverId' in FOR EACH clause is not a primary key column of table 'drivr'
6. ERROR at line 2:28 - Table 'drivers' in static aggregation is not connected to main table 'drivr'
7. ERROR at line 2:24 - Aggregation type 'AVG' cannot be used in string condition


SystemExit: 1

In [148]:
static_helper.convert_query("""
                PREDICT drivers.driverId
                FOR EACH drivers.driverId
                WHERE drivers.surname CONTAINS "l";
              """)


                PREDICT drivers.driverId
                FOR EACH drivers.driverId
                WHERE drivers.surname CONTAINS "l";
              
------PREDICT_START------
SELECT
    for_each.fk AS fk,
    main.comp_col
 AS label
FROM
    (SELECT
        driverId AS fk
    FROM
        (SELECT
        *
    FROM
        drivers unsorted_aggr_tbl
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------ID_DOT_ID_START------
        SELECT
            driverId AS fk,
            surname AS comp_col
        FROM
            drivers
        ------ID_DOT_ID_END------
    )
    WHERE
        comp_col LIKE '%l%'
    ------CONDITION_END------
    ) expr
    ON
        unsorted_aggr_tbl.driverId = expr.fk
    )
    
) for_each
LEFT JOIN
    (------ID_DOT_ID_START------
    SELECT
        driverId AS fk,
        driverId AS comp_col
    FROM
        drivers
    ------ID_DOT_ID_END------
) main
ON
    main.fk = for_each.fk
ORDER BY
    for_each.fk ASC
------

In [146]:
static_helper.convert_query("""
                PREDICT drivers.driverId
                FOR EACH drivers.driverId
                WHERE drivers.driverId > 5;
              """)


                PREDICT drivers.driverId
                FOR EACH drivers.driverId
                WHERE drivers.driverId > 5;
              
------PREDICT_START------
SELECT
    for_each.fk AS fk,
    main.comp_col
 AS label
FROM
    (SELECT
        driverId AS fk
    FROM
        (SELECT
        *
    FROM
        drivers unsorted_aggr_tbl
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------ID_DOT_ID_START------
        SELECT
            driverId AS fk,
            driverId AS comp_col
        FROM
            drivers
        ------ID_DOT_ID_END------
    )
    WHERE
        comp_col > 5
    ------CONDITION_END------
    ) expr
    ON
        unsorted_aggr_tbl.driverId = expr.fk
    )
    
) for_each
LEFT JOIN
    (------ID_DOT_ID_START------
    SELECT
        driverId AS fk,
        driverId AS comp_col
    FROM
        drivers
    ------ID_DOT_ID_END------
) main
ON
    main.fk = for_each.fk
ORDER BY
    for_each.fk ASC
------PREDICT_END---

In [9]:
static_helper.convert_query("""
                PREDICT AVG(drivers.driverId) > 10
                FOR EACH drivers.driverId;
              """)


                PREDICT AVG(drivers.driverId) > 10
                FOR EACH drivers.driverId;
              
SELECT
    *
FROM
  (------PREDICT_START------
SELECT
    for_each.fk AS fk,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (------FOR_EACH_START------
    SELECT
        driverId AS fk
    FROM
        drivers
    ------FOR_EACH_END------
) for_each
LEFT JOIN
    (------CONDITION_START------
    SELECT
        *
    FROM
        (------STAT_AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            AVG(aggr_tbl.driverId)
         AS comp_col,
        FROM
            drivers parent
        LEFT JOIN
            drivers aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        GROUP BY
            parent.driverId
        ------STAT_AGGREGATION_END------
    )
    WHERE
        comp_col > 10
    ------CONDITION_END------
) main
ON
    main.fk = for_each.fk
------PREDICT_END------
)
WH